In [10]:
import os
os.chdir('/container/mount/point')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import pickle

from scipy.cluster.hierarchy import linkage, leaves_list

from gglasso.helper.utils import sparsity
from gglasso.helper.model_selection import ebic

from utils.helper import transform_features, scale_array_by_diagonal
from utils.stability_selection import subsample
from utils.solver import ADMM_single

In [11]:
def drop_all_zero_rows_and_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Drop columns that are all zero, then rows that are all zero (two-step to be safe)."""
    df1 = df.loc[:, (df != 0).any(axis=0)]
    df2 = df1.loc[(df1 != 0).any(axis=1), :]
    return df2

### Import American Gut Project data

In [7]:
kora_matched_df = pd.read_csv("data/smoking_KORA_experiment.csv", index_col=0)
asv = pd.read_csv("data/filtered_count_table.csv", index_col=0)
simulated_outcomes = pd.read_csv("data/simulated_KORA_outcomes.csv", index_col=0)
taxa = pd.read_csv('data/taxonomy_clean.csv', index_col=0)
print(f"ASV table shape (features, samples): {asv.shape}")

# Sort ASV columns to match sample order in kora_matched_df
sample_order = list(kora_matched_df.index.astype(str))
ASV_table = asv.reindex(sample_order, axis=1)

taxa_dict = {}
for level in ['domain', 'phylum', 'class', 'order', 'family', 'genus', 'species']:
    df_level = ASV_table.join(taxa[level])
    df_level = df_level.groupby(level).sum()
    taxa_dict[level] = df_level
taxa_dict["ASVs"] = ASV_table

for level in taxa_dict.keys():
    print(f"{level} count table shape: {taxa_dict[level].shape}")

with open("data/taxa_dict.pkl", "wb") as f:
    pickle.dump(taxa_dict, f)

ASV table shape (features, samples): (1469, 436)
domain count table shape: (2, 436)
phylum count table shape: (9, 436)
class count table shape: (15, 436)
order count table shape: (41, 436)
family count table shape: (80, 436)
genus count table shape: (401, 436)
species count table shape: (1354, 436)
ASVs count table shape: (1469, 436)


In [25]:
# Choose level
level = "family"

# Get the aggregated count table (rows = taxa, cols = individuals)
counts = taxa_dict[level].copy()

# Ensure column labels are strings to match index types
counts.columns = counts.columns.astype(str)

# Build column lists for smokers / non-smokers (as strings)
smoker_cols = counts.columns.intersection(kora_matched_df.loc[kora_matched_df['smoking_bin'] == 0].index.astype(str))
nonsmoker_cols = counts.columns.intersection(kora_matched_df.loc[kora_matched_df['smoking_bin'] == 1].index.astype(str))

# Subset and clean
counts_smokers = drop_all_zero_rows_and_cols(counts.loc[:, smoker_cols])
counts_nonsmokers = drop_all_zero_rows_and_cols(counts.loc[:, nonsmoker_cols])

print(f"Smokers:  shape {counts_smokers.shape} (taxa x individuals)")
print(f"Non-smokers: shape {counts_nonsmokers.shape} (taxa x individuals)")

# Optional: sanity checks
assert (counts_smokers != 0).any(axis=0).all(), "Found an all-zero column in smokers after cleaning."
assert (counts_smokers != 0).any(axis=1).all(), "Found an all-zero row in smokers after cleaning."
assert (counts_nonsmokers != 0).any(axis=0).all(), "Found an all-zero column in non-smokers after cleaning."
assert (counts_nonsmokers != 0).any(axis=1).all(), "Found an all-zero row in non-smokers after cleaning."

### mCLR-transform of the counts
counts_smokers_mclr = transform_features(counts_smokers, transformation="mclr")
counts_nonsmokers_mclr = transform_features(counts_nonsmokers, transformation="mclr")

### calculate correlation
S_smoker = np.cov(counts_smokers_mclr, bias=True)
corr_smoker = scale_array_by_diagonal(S_smoker)

S_nonsmoker = np.cov(counts_nonsmokers_mclr, bias=True)
corr_nonsmoker = scale_array_by_diagonal(S_nonsmoker)


f_count_smoker = counts_smokers.T.copy()  # N x p
f_count_non_smoker = counts_nonsmokers.T.copy()  # N x p

family_names = list(f_count_smoker.columns)

Smokers:  shape (80, 218) (taxa x individuals)
Non-smokers: shape (80, 218) (taxa x individuals)


### Perform Subsampling

In [26]:
level = "family"
model = "sparse"
N = 10  # Number of subsamples
lambda1_range = np.logspace(0, -2, 10)

# Process both smoker and non-smoker cases
data_dict = dict()

for case, f_count in zip(["smoker", "non_smoker"], [f_count_non_smoker, f_count_smoker]):
    print(f"Processing {case} case")
    
    # Select count table according to selected taxonomic level
    counts = f_count.copy().T
    p, n = counts.shape
    print(f"Level {level}: \n Shape(p,N): {counts.shape}")

    sample_ids = list(map(str, matched_df[matched_df["W"] == 1].index))

    clr_counts = transform_features(counts, transformation="clr")
    subsamples = subsample(clr_counts.values, N)

    for i in range(subsamples.shape[2]):  # Loop through each subsample
        subsample_slice = subsamples[:, :, i]  # Access the i-th subsample
        print(f"Subsample {i+1}:")
        print(subsample_slice.shape)

    print(f"Print subsamples object: {subsamples.shape}")

    data_dict[case] = {
        "counts": counts,
        "clr_counts": clr_counts,
        "subsamples": subsamples
    }

Processing smoker case
Level family: 
 Shape(p,N): (80, 218)
Subsample 1:
(80, 147)
Subsample 2:
(80, 147)
Subsample 3:
(80, 147)
Subsample 4:
(80, 147)
Subsample 5:
(80, 147)
Subsample 6:
(80, 147)
Subsample 7:
(80, 147)
Subsample 8:
(80, 147)
Subsample 9:
(80, 147)
Subsample 10:
(80, 147)
Print subsamples object: (80, 147, 10)
Processing non_smoker case
Level family: 
 Shape(p,N): (80, 218)
Subsample 1:
(80, 147)
Subsample 2:
(80, 147)
Subsample 3:
(80, 147)
Subsample 4:
(80, 147)
Subsample 5:
(80, 147)
Subsample 6:
(80, 147)
Subsample 7:
(80, 147)
Subsample 8:
(80, 147)
Subsample 9:
(80, 147)
Subsample 10:
(80, 147)
Print subsamples object: (80, 147, 10)


In [27]:
test = pd.DataFrame(np.cov(data_dict["smoker"]["clr_counts"], bias=True))

# fig = px.imshow(test,
#                 color_continuous_scale='RdBu_r',
#                 labels={'color': 'Value'},
#                 title=f'Test: p={p}, n={n}',
#                 color_continuous_midpoint=0)

# # Customize layout for better visuals (optional)
# fig.update_layout(
#     width=850,
#     height=850,
#     xaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in S_hier.index.map(int)]),
#     yaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in S_hier.columns.map(int)])
# )

# fig.show()

off_diagonal_elements = test.values[np.triu_indices_from(test.values, k=1)]
max_off_diagonal_value = np.max(abs(off_diagonal_elements))
max_off_diagonal_value

3.4454178317503565

### Empirical covariance

In [28]:
cov_dict = {"smoker": dict(), "non_smoker": dict()}
corr_dict = {"smoker": dict(), "non_smoker": dict()}

for case, f_count in zip(["smoker", "non_smoker"], [f_count_non_smoker, f_count_smoker]):
    print(f"Processing {case} case")

    clr_counts = data_dict[case]["clr_counts"]
    
    # Empriical covariance matrix
    S = np.cov(clr_counts, bias=True)
    corr = scale_array_by_diagonal(S)
    if p != S.shape[0]:
        raise Exception("Check covariance shape!")
    
    cov_dict[case] = S
    corr_dict[case] = corr

Processing smoker case
Processing non_smoker case


In [35]:
### Hierarchical clustering of the correlation matrix
linkage_counts = linkage(cov_dict["smoker"], method='average')
ordered_indices = leaves_list(linkage_counts)

for case, f_count in zip(["smoker", "non_smoker"], [f_count_non_smoker, f_count_smoker]):
    print(f"Processing {case} case")

    # S = cov_dict[case]
    S = corr_dict[case]

    S_hier = pd.DataFrame(S, index=family_names, columns=family_names)
    S_hier = S_hier.iloc[ordered_indices, ordered_indices]

    fig = px.imshow(S_hier,
                color_continuous_scale='RdBu_r',
                labels={'color': 'Value'},
                title=f'Empirical covariance: p={p}, n={n} ({case})', 
                zmin=-1, 
                zmax=1)
                

    # Customize layout for better visuals (optional)
    fig.update_layout(
        width=1500,
        height=1500,
        xaxis=dict(tickmode='array', tickvals=list(range(len(S_hier.index))), ticktext=list(S_hier.index)),
        yaxis=dict(tickmode='array', tickvals=list(range(len(S_hier.columns))), ticktext=list(S_hier.columns))
    )

    fig.show()

    # fig.write_image(f"plots/png/covaraince_{case}.png")
    # fig.write_image(f"plots/pdf/covaraince_{case}.pdf")
    # fig.write_image(f"plots/svg/covaraince_{case}.svg")
    # fig.write_html(f"plots/html/covaraince_{case}.html")

    fig.write_image(f"plots/png/correlation_{case}_KORA_smoking.png")
    fig.write_image(f"plots/pdf/correlation_{case}_KORA_smoking.pdf")
    fig.write_image(f"plots/svg/correlation_{case}_KORA_smoking.svg")
    fig.write_html(f"plots/html/correlation_{case}_KORA_smoking.html")

Processing smoker case


Processing non_smoker case


## GGLasso solution

In [36]:
lambda1_range_smoker = np.logspace(0, -3, 10)
lambda1_range_non_smoker =  np.logspace(0, -3, 10)


rank_range = [2, 3, 4, 5, 6, 7, 8, 9, 10]
verbose = False
tol, rtol = 1e-5, 1e-5
beta = 0.05
mu1 = 1

low_rank_dict = {"smoker": dict(), "non_smoker": dict()}

for case in ["smoker", "non_smoker"]:
    print(f"Processing {case} case")

    control_counts = data_dict[case]["counts"]
    control_clr_counts = data_dict[case]["clr_counts"]
    subsamples = data_dict[case]["subsamples"]

    n_samples = control_clr_counts.shape[1]

    for r in rank_range:
        print(f"--------------     Rank: {r} --------------")

        sol_dict = dict()

        lambda1_range = lambda1_range_smoker if case == "smoker" else lambda1_range_non_smoker

        for lambda1 in lambda1_range:
            print(f"-------------- Lambda: {lambda1} --------------")

            sub_sol_dict = dict()

            ### Loop through each subsample
            for i in range(N):
                subsample_slice = subsamples[:, :, i]
                print(f"Subsample {i+1}:")

                S = np.cov(subsample_slice, bias=True)
                # S = scale_array_by_diagonal(S0)

                if p != S.shape[0]:
                    raise Exception("Check covariance shape!")

                ### Solve the sparse problem
                sub_sol, _ = ADMM_single(S=S, lambda1=lambda1, mu1=mu1, r=r, Omega_0=np.eye(p), 
                                verbose=verbose, latent=True, tol=tol, rtol=rtol)

                G = sub_sol['Theta'].astype(bool).astype(int)

                sub_sol_dict[f"subsample_{i+1}"] = {"Theta": sub_sol['Theta'],
                                                    "L": sub_sol['L'],
                                                    "Omega": sub_sol['Omega'],
                                                    "X": sub_sol["X"], "S": S,
                                                    "adjacency_matrix": G}

            # Store subsample solutions in sol_dict[lambda1]
            sol_dict[lambda1] = {"sub_samples": sub_sol_dict}

            estimates = list()

            for i in range(N):
                ### adjacency matrix (like in the paper) or estimate (like in R)?
                estimate = sol_dict[lambda1]["sub_samples"][f'subsample_{i+1}']["adjacency_matrix"]

                ### psi_Lambda_st in the paper notation
                estimates.append(estimate)

            ### theta_b_st_hat in the paper notation
            edge_average = np.mean(estimates, axis=0)

            ### the variance of a Bernoulli distribution (ksi_b_st in the paper notation)
            edge_instability = 2 * edge_average * (1 - edge_average)

            ### D_b in the paper notation
            total_instability = 2 * np.sum(edge_instability) / (p * (p - 1))

            # Add instability measure to sol_dict[lambda1]
            sol_dict[lambda1]["D_b"] = total_instability

        # Collect all instabilities for current mu1
        instabilities = [sol_dict[lambda1]["D_b"] for lambda1 in sol_dict if "D_b" in sol_dict[lambda1]]

        # Find the index of the largest lambda where total_instability <= stars_thresh
        indices = np.where(np.array(instabilities) <= beta)[0]

        if len(indices) > 0:
            opt_index = np.max(indices)
        else:
            opt_index = indices[0]  # pick the only one

        lambda_star = lambda1_range[opt_index]

        print(f"Optimal lambda: {lambda_star}")

        ### Do the refit of the model on the exactly the same empirical covariance matrix
        S = cov_dict[case]
        # S0 = np.cov(control_clr_counts, bias=True)
        # S = scale_array_by_diagonal(S0)
        # if p != S.shape[0]:
        #     raise Exception("Check covariance shape!")
        

        sol, _ = ADMM_single(S=S, lambda1=lambda_star, mu1=mu1, r=r, Omega_0=np.eye(p),
                        verbose=verbose, latent=True, tol=tol, rtol=rtol)
        
        print(f"Rank of the low-rank matrix: {np.linalg.matrix_rank(sol['L'])}")


        ebic_result = ebic(S, sol['Theta'], N=n_samples, gamma=0.5)

        sol_dict["refit"] = {"Theta": sol['Theta'], "Omega": sol['Omega'], "X": sol['X'],
                            "L": sol["L"], "S": S, "lambda_star": lambda_star, "eBIC": ebic_result,
                            "adjacency_matrix": sol['Theta'].astype(bool).astype(int)}

        low_rank_dict[case][r] = sol_dict

Processing smoker case
--------------     Rank: 2 --------------
-------------- Lambda: 1.0 --------------
Subsample 1:
ADMM terminated after 44 iterations with status: optimal.
Subsample 2:
ADMM terminated after 42 iterations with status: optimal.
Subsample 3:
ADMM terminated after 45 iterations with status: optimal.
Subsample 4:
ADMM terminated after 45 iterations with status: optimal.
Subsample 5:
ADMM terminated after 38 iterations with status: optimal.
Subsample 6:
ADMM terminated after 45 iterations with status: optimal.
Subsample 7:
ADMM terminated after 45 iterations with status: optimal.
Subsample 8:
ADMM terminated after 42 iterations with status: optimal.
Subsample 9:
ADMM terminated after 46 iterations with status: optimal.
Subsample 10:
ADMM terminated after 41 iterations with status: optimal.
-------------- Lambda: 0.4641588833612779 --------------
Subsample 1:
ADMM terminated after 44 iterations with status: optimal.
Subsample 2:
ADMM terminated after 40 iterations with 

KeyboardInterrupt: 

### Empirical covariance

In [296]:
selected_rank = 10

for case in ["smoker", "non_smoker"]:
    gg_S = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["S"], index=family_names, columns=family_names)
    gg_S = gg_S.iloc[ordered_indices, ordered_indices]

    fig = px.imshow(gg_S,
                    color_continuous_scale='RdBu_r',
                    labels={'color': 'Value'},
                    title=f'GGLasso: S_hat ({case})')

    fig.update_layout(
        width=850,
        height=850,
        xaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in gg_S.index.map(int)]),
        yaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in gg_S.columns.map(int)])
    )

    fig.show()


### Sparse part

In [301]:
selected_rank = 10

for case in ["smoker", "non_smoker"]:
    gg_Theta = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["Theta"], index=family_names, columns=family_names)
    gg_Theta = gg_Theta.iloc[ordered_indices, ordered_indices]
    SP = np.round(sparsity(gg_Theta), 4)

    fig = px.imshow(gg_Theta,
                    color_continuous_scale='RdBu_r',
                    labels={'color': 'Value'},
                    title=f'GGLasso: Theta, SP={SP} ({case})',
                    color_continuous_midpoint=0)

    fig.update_layout(
        width=850,
        height=850,
        xaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in gg_Theta.index.map(int)]),
        yaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in gg_Theta.columns.map(int)])
    )

    fig.show()

    fig.write_image(f"plots/png/gg_sparse_part_{case}.png")
    fig.write_image(f"plots/pdf/gg_sparse_part_{case}.pdf")
    fig.write_image(f"plots/svg/gg_sparse_part_{case}.svg")
    fig.write_html(f"plots/html/gg_sparse_part_{case}.html")


### Low-rank part

In [302]:
selected_rank = 10

for case in ["smoker", "non_smoker"]:
    gg_L = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["L"], index=family_names, columns=family_names)
    gg_L = gg_L.iloc[ordered_indices, ordered_indices]
    rank_gg_L = np.linalg.matrix_rank(gg_L)
    print(f"Rank of the low-rank matrix from GGLasso ({case}): {rank_gg_L}")

    fig = px.imshow(gg_L,
                    color_continuous_scale='RdBu_r',  # Adjust the color scale
                    labels={'color': 'Value'},          # Label for color bar
                    title=f'GGLasso: L (rank={rank_gg_L}) ({case})',
                    color_continuous_midpoint=0)

    # Customize layout for better visuals (optional)
    fig.update_layout(
        width=850,
        height=850,
        xaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in gg_L.index.map(int)]),
        yaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in gg_L.columns.map(int)])
    )

    fig.show()

    fig.write_image(f"plots/png/gg_lowrank_part_{case}.png")
    fig.write_image(f"plots/pdf/gg_lowrank_part_{case}.pdf")
    fig.write_image(f"plots/svg/gg_lowrank_part_{case}.svg")
    fig.write_html(f"plots/html/gg_lowrank_part_{case}.html")


Rank of the low-rank matrix from GGLasso (smoker): 10


Rank of the low-rank matrix from GGLasso (non_smoker): 10


### Likelihood

In [307]:
for case in ["smoker", "non_smoker"]:
    gg_Omega = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["Omega"], index=family_names, columns=family_names)
    gg_Omega = gg_Omega.iloc[ordered_indices, ordered_indices]

    fig = px.imshow(gg_Omega,
                    color_continuous_scale='RdBu_r',  
                    labels={'color': 'Value'},
                    title=f'GGLasso: Omega ({case})',
                    color_continuous_midpoint=0)

    # Customize layout for better visuals (optional)
    fig.update_layout(
        width=850,
        height=850,
        xaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in gg_Omega.index.map(int)]),
        yaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in gg_Omega.columns.map(int)])
    )

    fig.show()

    fig.write_image(f"plots/png/gg_omega_part_{case}.png")
    fig.write_image(f"plots/pdf/gg_omega_part_{case}.pdf")
    fig.write_image(f"plots/svg/gg_omega_part_{case}.svg")
    fig.write_html(f"plots/html/gg_omega_part_{case}.html")

In [317]:
for case in ["smoker", "non_smoker"]:
    gg_S = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["S"], index=family_names, columns=family_names)
    gg_Omega = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["Omega"], index=family_names, columns=family_names)

    I = gg_Omega.dot(gg_S)
    I_reoder = I.iloc[ordered_indices, ordered_indices]

    fig = px.imshow(I_reoder,
                    color_continuous_scale='RdBu_r',  
                    labels={'color': 'Value'},
                    title=f'GGLasso: Omega x S_hat ({case})',
                    color_continuous_midpoint=0)

    # Customize layout for better visuals (optional)
    fig.update_layout(
        width=850,
        height=850,
        xaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in I_reoder.index.map(int)]),
        yaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in I_reoder.columns.map(int)])
    )

    fig.show()

    fig.write_image(f"plots/png/gg_identity_{case}.png")
    fig.write_image(f"plots/pdf/gg_identity_{case}.pdf")
    fig.write_image(f"plots/svg/gg_identity_{case}.svg")
    fig.write_html(f"plots/html/gg_identity_{case}.html")

In [305]:
for case in ["smoker", "non_smoker"]:
    ranks = list(low_rank_dict[case].keys())
    sparsities = [sparsity(low_rank_dict[case][rank]["refit"]["Theta"]) for rank in ranks]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ranks, y=sparsities, mode='lines+markers', name=f'Sparsity ({case})'))

    fig.update_layout(
        title=f'Sparsity vs Rank ({case})',
        xaxis_title='Rank',
        yaxis_title='Sparsity',
        template='plotly_white'
    )

    fig.show()

    for key in low_rank_dict[case].keys():
        print(f"Case: {case}, Rank: {key}")
        print(low_rank_dict[case][key]["refit"]["eBIC"])

Case: smoker, Rank: 2
20207.537874776786
Case: smoker, Rank: 3
20258.241035384952
Case: smoker, Rank: 4
20300.927344903077
Case: smoker, Rank: 5
20408.567815173537
Case: smoker, Rank: 6
20645.41031743037
Case: smoker, Rank: 7
20833.135651282973
Case: smoker, Rank: 8
21131.24875271713
Case: smoker, Rank: 9
21576.386128862967
Case: smoker, Rank: 10
21623.91386476948


Case: non_smoker, Rank: 2
22910.944246933665
Case: non_smoker, Rank: 3
23149.362739371358
Case: non_smoker, Rank: 4
23439.61556269154
Case: non_smoker, Rank: 5
23465.905740726583
Case: non_smoker, Rank: 6
24368.63073772679
Case: non_smoker, Rank: 7
24892.69552446395
Case: non_smoker, Rank: 8
25546.153964214172
Case: non_smoker, Rank: 9
25923.921247433198
Case: non_smoker, Rank: 10
26540.191588057245


### SVD of both solutions

In [306]:
for case in ["smoker", "non_smoker"]:
    se_L = pd.read_csv(f'data/AGP/low_rank_{case}.csv', sep=',', index_col=0)
    gg_L = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["L"], index=family_names, columns=family_names)

    # Perform Singular Value Decomposition (SVD) on both matrices
    U_se, s_se, Vt_se = np.linalg.svd(se_L)
    U_gg, s_gg, Vt_gg = np.linalg.svd(gg_L)

    # Compare eigenvalues
    print(f"Eigenvalues of se_L ({case}):")
    print(s_se)
    print(f"\nEigenvalues of gg_L ({case}):")
    print(s_gg)
    print(f"\nDifference between SE and GG ({case}):")
    print(np.linalg.norm(se_L.values - gg_L.values))
    
    # Plot the spectrum of eigenvalues
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=list(range(len(s_se))), y=s_se, mode='lines+markers', name=f'SE Eigenvalues ({case})'))
    fig.add_trace(go.Scatter(x=list(range(len(s_gg))), y=s_gg, mode='lines+markers', name=f'GG Eigenvalues ({case})'))

    fig.update_layout(
        title=f'Eigenvalue Spectrum Comparison ({case})',
        xaxis_title='Eigenvalue Index',
        yaxis_title='Eigenvalue',
        template='plotly_white'
    )

    fig.show()

    fig.write_image(f"plots/png/spectrum_plot_{case}.png")
    fig.write_image(f"plots/pdf/spectrum_plot_{case}.pdf")
    fig.write_image(f"plots/svg/spectrum_plot_{case}.svg")
    fig.write_html(f"plots/html/spectrum_plot_{case}.html")

Eigenvalues of se_L (smoker):
[4.81950257e-01 3.17211505e-01 2.39490164e-01 2.17153912e-01
 1.40416947e-01 9.41347012e-02 7.24883264e-02 6.73928338e-02
 3.08306066e-02 5.36903359e-03 4.28057817e-16 3.82752158e-16
 3.57328332e-16 3.34684189e-16 3.26563259e-16 2.95731216e-16
 2.82656519e-16 2.80128464e-16 2.65834802e-16 2.57032684e-16
 2.28009502e-16 2.26526398e-16 2.08508392e-16 1.83219080e-16
 1.76174948e-16 1.66337058e-16 1.61784510e-16 1.49742126e-16
 1.26011131e-16 1.11057731e-16 1.06098174e-16 9.51511228e-17
 9.26487331e-17 8.72575925e-17 5.73626500e-17 5.40587300e-17
 3.95695689e-17 2.57510519e-17 2.32278107e-17 1.59413943e-17]

Eigenvalues of gg_L (smoker):
[6.96018286e-01 4.02849004e-01 3.21254756e-01 3.10492963e-01
 2.43888799e-01 2.01032171e-01 1.75639735e-01 1.36967329e-01
 7.88970206e-02 1.09484408e-02 5.08030863e-17 5.05222526e-17
 5.05222526e-17 5.05222526e-17 5.05222526e-17 5.05222526e-17
 5.05222526e-17 5.05222526e-17 5.05222526e-17 5.05222526e-17
 5.05222526e-17 5.05222

Eigenvalues of se_L (non_smoker):
[4.86107316e-01 2.58969619e-01 2.45004652e-01 1.29174767e-01
 8.11749284e-02 5.67155277e-02 5.34183753e-02 3.03558096e-02
 2.61841384e-02 8.02153982e-03 4.73086299e-16 4.31750028e-16
 4.25039652e-16 3.86996688e-16 3.56778755e-16 3.40659694e-16
 3.26065441e-16 3.08015914e-16 2.85426458e-16 2.78331056e-16
 2.59240545e-16 2.46870231e-16 2.38536530e-16 2.28439343e-16
 2.05338178e-16 1.80378509e-16 1.68985546e-16 1.48929517e-16
 1.24268233e-16 1.21063045e-16 1.11132283e-16 1.06250611e-16
 9.31854329e-17 8.77561739e-17 6.30947167e-17 5.94350977e-17
 4.57180903e-17 4.18045432e-17 2.03577125e-17 1.06546310e-17]

Eigenvalues of gg_L (non_smoker):
[6.60045637e-01 4.68170613e-01 3.45892020e-01 3.04570994e-01
 2.72013578e-01 2.26090298e-01 1.75673348e-01 8.14889289e-02
 5.62308122e-02 2.31666710e-02 4.31982233e-17 4.31982233e-17
 4.31982233e-17 4.31982233e-17 4.31982233e-17 4.31982233e-17
 4.31982233e-17 4.31982233e-17 4.31982233e-17 4.31982233e-17
 4.31982233e-17

In [328]:
for case, se_Theta in zip(["smoker", "non_smoker"], [se_Theta_smoker, se_Theta_nonsmoker]):
    print(f"{case} case")
    ### SPIEC-EASI
    se_Theta.index, se_Theta.columns = family_names, family_names
    se_Theta = se_Theta.iloc[ordered_indices, ordered_indices]
    se_SP = np.round(sparsity(se_Theta), 4)
    print(f"SE Theta Sparsity {se_SP}")

    se_L = pd.read_csv(f'data/AGP/low_rank_{case}.csv', sep=',', index_col=0)
    se_L.index, se_L.columns = family_names, family_names
    se_L = se_L.iloc[ordered_indices, ordered_indices]

    se_Omega = se_Theta - se_L

    ### GGLasso
    gg_Theta = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["Theta"], index=family_names, columns=family_names)
    gg_Theta = gg_Theta.iloc[ordered_indices, ordered_indices]
    gg_Omega = pd.DataFrame(low_rank_dict[case][selected_rank]["refit"]["Omega"], index=family_names, columns=family_names)
    gg_Omega = gg_Omega.iloc[ordered_indices, ordered_indices]
    
    gg_SP = np.round(sparsity(gg_Theta), 4)
    print(f"GG Theta Sparsity {gg_SP} \n")

    diff_Omega = se_Omega - gg_Omega
    diff_Theta = se_Theta - gg_Theta

    fig_omega = px.imshow(diff_Omega,
                          color_continuous_scale='RdBu_r',  
                          labels={'color': 'Value'},
                          title=f'Difference (SE-GG): Omega ({case})',
                          color_continuous_midpoint=0)

    fig_omega.update_layout(
        width=850,
        height=850,
        xaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in diff_Omega.index.map(int)]),
        yaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in diff_Omega.columns.map(int)])
    )

    fig_omega.show()

    fig_omega.write_image(f"plots/png/difference_omega_{case}.png")
    fig_omega.write_image(f"plots/pdf/difference_omega_{case}.pdf")
    fig_omega.write_image(f"plots/svg/difference_omega_{case}.svg")
    fig_omega.write_html(f"plots/html/difference_omega_{case}.html")

    fig_theta = px.imshow(diff_Theta,
                          color_continuous_scale='RdBu_r',  
                          labels={'color': 'Value'},
                          title=f'Difference (SE-GG): Theta ({case})',
                          color_continuous_midpoint=0)

    fig_theta.update_layout(
        width=850,
        height=850,
        xaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in diff_Theta.index.map(int)]),
        yaxis=dict(tickmode='array', tickvals=list(range(len(names_dict))), ticktext=[names_dict[idx] for idx in diff_Theta.columns.map(int)])
    )

    fig_theta.show()

    fig_theta.write_image(f"plots/png/difference_theta_{case}.png")
    fig_theta.write_image(f"plots/pdf/difference_theta_{case}.pdf")
    fig_theta.write_image(f"plots/svg/difference_theta_{case}.svg")
    fig_theta.write_html(f"plots/html/difference_theta_{case}.html")

smoker case
SE Theta Sparsity 0.0167
GG Theta Sparsity 0.0026 



non_smoker case
SE Theta Sparsity 0.0615
GG Theta Sparsity 0.0154 

